## Data Loading

In [35]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, TimeDistributed, Reshape, Lambda, Concatenate
from tensorflow.keras.layers import Softmax, Multiply
import tensorflow as tf
import json
import glob

In [36]:
# Load all JSON files from the Data subfolder and assign a sequential userID per file
data_dir = os.path.join('Data')
json_files = sorted(glob.glob(os.path.join(data_dir, '*.json')))
all_samples = []

for user_id, file in enumerate(json_files, start=1):
    with open(file, "r") as f:
        raw = json.load(f)

    if isinstance(raw, list):
        for session in raw:
            for sample in session['samples']:
                sample['userID'] = user_id
                all_samples.append(sample)
    else:
        for sample in raw['samples']:
            sample['userID'] = user_id
            all_samples.append(sample)

print(f"Loaded {len(json_files)} JSON files from '{data_dir}'")
print(f"Total samples: {len(all_samples)}")

# Extract samples
df_data = pd.DataFrame(all_samples)
df = pd.json_normalize(all_samples)

df_data.set_index('time', inplace=True)

Loaded 16 JSON files from 'Data'
Total samples: 242186


In [37]:
# Filter valid samples
df_data = df_data[df_data['user'].apply(lambda x: isinstance(x, dict)) & df_data['pois'].apply(lambda x: isinstance(x, list))]
print(f"Filtered data length: {len(df_data)}")

Filtered data length: 242186


In [38]:
print(df_data['user'].apply(type).value_counts())
print()
print(df_data['pois'].apply(type).value_counts())
print()
print(df_data.columns)

user
<class 'dict'>    242186
Name: count, dtype: int64

pois
<class 'list'>    242186
Name: count, dtype: int64

Index(['env', 'trial', 'user', 'pois', 'userID'], dtype='object')


## Flatten and Combine Features

In [39]:
def flatten_pois_safe(pois_list, user_yaw, max_pois=5):
    keys = ['dist', 'sin_angle', 'cos_angle', 'is_in_front', 'dist_weight']

    if not isinstance(pois_list, list):
        return {f'poi_{i}_{k}': 0 for i in range(max_pois) for k in keys}

    pois_list = sorted(pois_list, key=lambda x: x['distance'])

    flat = {}

    for i in range(max_pois):
        if i < len(pois_list):
            dist = pois_list[i]['distance']
            angle = pois_list[i]['angle']

            # Convert POI angle into user-relative frame for this timestep
            rel_angle = np.arctan2(
                np.sin(angle - user_yaw),
                np.cos(angle - user_yaw)
            )

            flat[f'poi_{i}_dist'] = dist
            flat[f'poi_{i}_sin_angle'] = np.sin(rel_angle)
            flat[f'poi_{i}_cos_angle'] = np.cos(rel_angle)
            flat[f'poi_{i}_is_in_front'] = 1 if abs(rel_angle) <= np.pi / 2 else 0
            flat[f'poi_{i}_dist_weight'] = 1 / (dist + 0.001)
        else:
            for k in keys:
                flat[f'poi_{i}_{k}'] = 0

    return flat

In [40]:
# Flatten user
user_df_full = pd.json_normalize(df_data['user'])
user_df_full.columns = ['x', 'z', 'vx', 'vz', 'yaw']

# Rotate velocities to local coordinate frame
user_df_full['vx_local'] = (
    user_df_full['vx'] * np.cos(user_df_full['yaw']) +
    user_df_full['vz'] * np.sin(user_df_full['yaw'])
)

user_df_full['vz_local'] = (
    -user_df_full['vx'] * np.sin(user_df_full['yaw']) +
    user_df_full['vz'] * np.cos(user_df_full['yaw'])
)

# Use original yaw here to make POIs user-relative
pois_df = pd.DataFrame([
    flatten_pois_safe(pois, yaw)
    for pois, yaw in zip(df_data['pois'], user_df_full['yaw'])
])

# Final user features
user_df = user_df_full[['x', 'z', 'vx_local', 'vz_local', 'yaw']]
user_df.columns = ['x', 'z', 'vx', 'vz', 'yaw']

In [41]:
df_final = pd.concat([
    df_data.reset_index()[['time', 'env', 'trial', 'userID']].reset_index(drop=True),
    user_df.reset_index(drop=True),
    pois_df.reset_index(drop=True)
], axis=1)

df_final = df_final.dropna()

## Define Evaluation Metrics

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate_predictions(name, y_true, preds):
    mae_x = mean_absolute_error(y_true[:, 0], preds[:, 0])
    mae_z = mean_absolute_error(y_true[:, 1], preds[:, 1])

    rmse_x = np.sqrt(mean_squared_error(y_true[:, 0], preds[:, 0]))
    rmse_z = np.sqrt(mean_squared_error(y_true[:, 1], preds[:, 1]))

    mae_overall = (mae_x + mae_z) / 2
    rmse_overall = (rmse_x + rmse_z) / 2

    mde = np.mean(np.linalg.norm(preds - y_true, axis=1))

    pred_angles = np.arctan2(preds[:, 1], preds[:, 0])
    actual_angles = np.arctan2(y_true[:, 1], y_true[:, 0])

    angle_diff = pred_angles - actual_angles
    angle_diff = (angle_diff + np.pi) % (2 * np.pi) - np.pi

    actual_mag = np.linalg.norm(y_true, axis=1)
    valid = actual_mag > 0.05

    angular_error = np.mean(np.abs(angle_diff[valid]))

    print(f"\n{name}")
    print(f"MAE X: {mae_x:.4f}, Z: {mae_z:.4f}")
    print(f"RMSE X: {rmse_x:.4f}, Z: {rmse_z:.4f}")
    print(f"Overall MAE: {mae_overall:.4f}")
    print(f"Overall RMSE: {rmse_overall:.4f}")
    print(f"MDE: {mde:.4f}")
    print(f"Angular Error: {angular_error:.4f} radians")

    return {
        "Model": name,
        "MAE": mae_overall,
        "RMSE": rmse_overall,
        "MDE": mde,
        "Angular_Error": angular_error
    }

## Defining Sequences

In [9]:
 # Create sequences of 50 timesteps for LSTM, predicting displacement 50 timesteps ahead
# from tokenize import group


def create_sequences_grouped(df, seq_length=50, prediction_horizon=50):
    X, y = [], []
    
    for _, group in df.groupby(['env', 'trial']):
        
        group = group.reset_index(drop=True)
        
        # Drop non-feature columns
        positions = group[['x', 'z']].values

        features = group.drop(
            columns=['env', 'trial', 'time', 'userID', 'x', 'z']
        ).values
        
        for i in range(len(features) - seq_length - prediction_horizon + 1):
            X.append(features[i:i+seq_length])
            # Predict displacement: future position - current position
            current_pos = positions[i + seq_length - 1]
            future_pos = positions[i + seq_length + prediction_horizon - 1]
            displacement = future_pos - current_pos
            y.append(displacement)
    
    X = np.array(X)
    y = np.array(y)
    
    # Apply yaw normalization and rotate target displacement into sequence-relative frame
    for i in range(len(X)):
        mean_theta = np.mean(X[i, :, 2])
        cos_t = np.cos(mean_theta)
        sin_t = np.sin(mean_theta)

        # Normalize yaw only
        for j in range(seq_length):
            yaw = X[i, j, 2]
            X[i, j, 2] = np.arctan2(
                np.sin(yaw - mean_theta),
                np.cos(yaw - mean_theta)
            )

        # Rotate target displacement from world frame into sequence-relative frame
        dx = y[i, 0]
        dz = y[i, 1]

        y[i, 0] = dx * cos_t + dz * sin_t
        y[i, 1] = -dx * sin_t + dz * cos_t
    return X, y

In [10]:
HISTORY_LENGTH = 50  # 2.5 seconds of history
PREDICTION_HORIZON = 50  # predict 2.5 seconds ahead

## Round 1: Exclude Environment A

In [ ]:
# Splitting by environment
test_env = 'A'

train_df = df_final[df_final['env'] != test_env]
test_df  = df_final[df_final['env'] == test_env]

In [ ]:
# Generate timestep sequences
X_train, y_train = create_sequences_grouped(train_df, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
X_test, y_test = create_sequences_grouped(test_df, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

### POI Model Construction

In [ ]:
# Build an LSTM model that encodes each POI via a shared MLP and mean-pools POI context
inputs = Input(shape=(HISTORY_LENGTH, 28), name='trajectory_input')

# Split user trajectory and POI features (skip userID at index 0)
user_features = Lambda(lambda x: x[:, :, :3], name='user_features')(inputs)  # vx, vz, yaw

# Suppose you have 5 POIs, each with 1+ features (here, 9 features for all POIs)
# Reshape to (batch, HISTORY_LENGTH, 5, 1+poi_feat_dim) if needed
poi_features = Lambda(lambda x: x[:, :, 3:], name='poi_raw')(inputs)
poi_reshaped = Reshape((HISTORY_LENGTH, 5, 5), name='poi_reshape')(poi_features)  # Adjust as needed

# Shared MLP for each POI
poi_encoded = TimeDistributed(TimeDistributed(Dense(32, activation='relu')), name='poi_mlp_1')(poi_reshaped)
poi_encoded = TimeDistributed(TimeDistributed(Dense(16, activation='relu')), name='poi_mlp_2')(poi_encoded)

# Distance-biased attention pooling across the 5 POIs
poi_scores = TimeDistributed(
    TimeDistributed(Dense(1)),
    name='learned_poi_scores'
)(poi_encoded)

poi_distances = Lambda(
    lambda x: x[:, :, :, 0:1],
    name='poi_distances'
)(poi_reshaped)

distance_bias = Lambda(
    lambda d: -d,
    name='distance_bias'
)(poi_distances)

combined_scores = Lambda(
    lambda x: x[0] + x[1],
    name='combined_attention_scores'
)([poi_scores, distance_bias])

poi_weights = Softmax(axis=2, name='poi_attention_weights')(combined_scores)

weighted_pois = Multiply(name='weighted_pois')([poi_encoded, poi_weights])

poi_context = Lambda(
    lambda x: tf.reduce_sum(x, axis=2),
    name='poi_attention_pool'
)(weighted_pois)

# Concatenate the POI context with user trajectory features
fused = Concatenate(axis=-1, name='trajectory_poi_fusion')([user_features, poi_context])

# Process fused sequence through LSTM stack
x = LSTM(64, return_sequences=True, name='lstm_1')(fused)
x = LSTM(64, name='lstm_2')(x)
x = Dropout(0.3, name='dropout')(x)
outputs = Dense(2, name='position_output')(x)

# Build model
model = tf.keras.Model(inputs=inputs, outputs=outputs, name='poicontext_lstm')

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### POI Model Training

In [ ]:
# Train the model
model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_test, y_test)
)

In [61]:
model.save("Exclude_Env_A_POI_Model.keras");

### POI Model Testing

In [ ]:
# make predictions
preds = model.predict(X_test)

In [49]:
# Baseline 1: zero displacement
preds_zero = np.zeros_like(y_test)

# Baseline 2: constant velocity
dt = 0.05
horizon_seconds = PREDICTION_HORIZON * dt

last_v = X_test[:, -1, :2]  # final vx, vz from input sequence
preds_cv = last_v * horizon_seconds

In [ ]:
results = []

results.append(evaluate_predictions("POI Attention LSTM", y_test, preds))
results.append(evaluate_predictions("Zero Displacement Baseline", y_test, preds_zero))
results.append(evaluate_predictions("Constant Velocity Baseline", y_test, preds_cv))


POI Attention LSTM
MAE X: 1.0239, Z: 1.0380
RMSE X: 1.3902, Z: 1.4283
Overall MAE: 1.0310
Overall RMSE: 1.4092
MDE: 1.6197
Angular Error: 0.9026 radians

Zero Displacement Baseline
MAE X: 1.2922, Z: 1.2998
RMSE X: 1.5767, Z: 1.5882
Overall MAE: 1.2960
Overall RMSE: 1.5824
MDE: 2.0347
Angular Error: 1.5827 radians

Constant Velocity Baseline
MAE X: 1.5360, Z: 1.6443
RMSE X: 2.0460, Z: 2.3221
Overall MAE: 1.5902
Overall RMSE: 2.1841
MDE: 2.4954
Angular Error: 1.2661 radians


### Visualize Predictions

In [ ]:
# Plot actual vs predicted for X
plt.figure(figsize=(10,6))

# Actual
plt.plot(y_test[:,0], label='Actual X')
# Predicted
plt.plot(preds[:,0], label='Predicted X')

plt.legend()
plt.title("Prediction vs Actual (X)")
plt.show()

In [53]:
# Select one trial from environment D
selected_trial = test_df['trial'].unique()[0]
test_df_single = test_df[test_df['trial'] == selected_trial]

# Create sequences for this single trial (each predicts 50 timesteps ahead)
X_test_single, y_test_single = create_sequences_grouped(test_df_single, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
preds_single = model.predict(X_test_single)

# Get actual positions
actual_x = test_df_single['x'].values
actual_z = test_df_single['z'].values

# Current positions (end of each input sequence)
current_x = X_test_single[:, -1, 0]
current_z = X_test_single[:, -1, 1]

# Predicted next positions (50 timesteps ahead)
pred_x = preds_single[:, 0]
pred_z = preds_single[:, 1]

# Calculate POI positions (at the user's position when closest to each POI)
poi_x_list = []
poi_z_list = []
for poi in range(5):
    dist_col = f'poi_{poi}_dist'
    min_dist_idx = test_df_single[dist_col].idxmin()
    poi_x = test_df_single.loc[min_dist_idx, 'x']
    poi_z = test_df_single.loc[min_dist_idx, 'z']
    poi_x_list.append(poi_x)
    poi_z_list.append(poi_z)

# Plot interactive map
import plotly.graph_objects as go

fig = go.Figure()

# Add actual path
fig.add_trace(go.Scatter(x=actual_x, y=actual_z, mode='lines+markers', name='Actual Path', 
                         line=dict(color='blue', width=2), marker=dict(size=4)))

# Add start and end markers
fig.add_trace(go.Scatter(x=[actual_x[0]], y=[actual_z[0]], mode='markers', name='Start', 
                         marker=dict(color='green', size=10, symbol='star')))
fig.add_trace(go.Scatter(x=[actual_x[-1]], y=[actual_z[-1]], mode='markers', name='End', 
                         marker=dict(color='purple', size=10, symbol='star')))

# Add predicted jumps (50 timesteps ahead from each sequence end)
for i in range(len(current_x)):
    fig.add_trace(go.Scatter(x=[current_x[i], pred_x[i]], y=[current_z[i], pred_z[i]], 
                             mode='lines', name='Predicted Jump (2.5s ahead)' if i == 0 else '', 
                             line=dict(color='red', width=2, dash='dash'), showlegend=i==0))

# Add POIs
fig.add_trace(go.Scatter(x=poi_x_list, y=poi_z_list, mode='markers', name='POIs', 
                         marker=dict(color='green', size=6, symbol='x')))

fig.update_layout(
    title=f'Map of Actual and Predicted User Walking Paths (Environment D, Trial {selected_trial}) - Predicting 2.5s Ahead',
    xaxis_title='X Position',
    yaxis_title='Z Position',
    width=800,
    height=600
)

fig.show()

### Evaluate POI Model Performance

In [54]:
# Analyze POI data validity
import matplotlib.pyplot as plt

poi_dist_cols = [col for col in df_final.columns if 'poi' in col and 'dist' in col and 'weight' not in col]
poi_sin_cols = [col for col in df_final.columns if 'poi' in col and 'sin_angle' in col]
poi_cos_cols = [col for col in df_final.columns if 'poi' in col and 'cos_angle' in col]

print("POI Distance Statistics:")
print(df_final[poi_dist_cols].describe())

print("\nPOI Sin Angle Statistics:")
print(df_final[poi_sin_cols].describe())

print("\nPOI Cos Angle Statistics:")
print(df_final[poi_cos_cols].describe())

# Check correlation between yaw and POI features
yaw_col = 'yaw'

corrs_yaw_sin = df_final[[yaw_col] + poi_sin_cols].corr()
corrs_yaw_cos = df_final[[yaw_col] + poi_cos_cols].corr()

print("\nCorrelation between yaw and POI sin(angle):")
print(corrs_yaw_sin[yaw_col])

print("\nCorrelation between yaw and POI cos(angle):")
print(corrs_yaw_cos[yaw_col])

# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, col in enumerate(poi_dist_cols[:3]):
    axes[0, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[0, i].set_title(f'{col} Distribution')
    axes[0, i].set_xlabel('Distance')
    axes[0, i].set_ylabel('Frequency')

for i, col in enumerate(poi_sin_cols[:3]):
    axes[1, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[1, i].set_title(f'{col} Distribution')
    axes[1, i].set_xlabel('sin(angle)')
    axes[1, i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Check ranges
print(f"\nSample sin(angle): {df_final[poi_sin_cols[0]].head()}")
print(f"sin(angle) range: {df_final[poi_sin_cols[0]].min():.3f} to {df_final[poi_sin_cols[0]].max():.3f}")

print(f"\nSample cos(angle): {df_final[poi_cos_cols[0]].head()}")
print(f"cos(angle) range: {df_final[poi_cos_cols[0]].min():.3f} to {df_final[poi_cos_cols[0]].max():.3f}")

print(f"\nYaw range: {df_final[yaw_col].min():.3f} to {df_final[yaw_col].max():.3f}")

In [ ]:
# Check correlations between POI features and target positions
poi_cols = [col for col in df_final.columns if 'poi' in col]
target_cols = ['x', 'z']

corrs = df_final[poi_cols + target_cols].corr()
print("Correlations between POI features and positions:")
print(corrs.loc[poi_cols, target_cols])

# Also check with velocities and yaw
all_features = ['x', 'z', 'vx', 'vz', 'yaw'] + poi_cols
corrs_full = df_final[all_features].corr()
print("\nFull feature correlations with x and z:")
print(corrs_full.loc[all_features, ['x', 'z']])

# Check if POI features have predictive power
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Simple linear regression for each POI feature to x
for col in poi_cols:
    X_feat = df_final[[col]]
    y_x = df_final['x']
    model_x = LinearRegression().fit(X_feat, y_x)
    r2_x = r2_score(y_x, model_x.predict(X_feat))
    
    y_z = df_final['z']
    model_z = LinearRegression().fit(X_feat, y_z)
    r2_z = r2_score(y_z, model_z.predict(X_feat))
    
    print(f"{col}: R² with x = {r2_x:.4f}, with z = {r2_z:.4f}")

### Baseline Model Construction

In [56]:
# Features without POIs: vx, vz, yaw (3 features, indices 0:3)
X_train_no_poi = X_train[:, :, :3]
X_test_no_poi = X_test[:, :, :3]

model_no_poi = Sequential([
    LSTM(64, input_shape=(HISTORY_LENGTH, 3), return_sequences=True),
    LSTM(64),
    Dropout(0.3),
    Dense(2)
])

model_no_poi.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### Baseline Model Training

In [57]:
#Training model without POI features
history_no_poi = model_no_poi.fit(
    X_train_no_poi, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_no_poi, y_test),
    verbose=0
)

In [64]:
model_no_poi.save("Exclude_Env_A_Baseline_Model.keras");

### Baseline Model Testing

In [58]:
preds_no_poi = model_no_poi.predict(X_test_no_poi)

results.append(evaluate_predictions("No-POI LSTM Baseline", y_test, preds_no_poi))

results_df = pd.DataFrame(results)
print(results_df)

1411/1411 [==============================] - 11s 8ms/step

No-POI LSTM Baseline
MAE X: 1.1863, Z: 1.1828
RMSE X: 1.5712, Z: 1.5858
Overall MAE: 1.1845
Overall RMSE: 1.5785
MDE: 1.8630
Angular Error: 1.0599 radians
                        Model       MAE      RMSE       MDE  Angular_Error
0          POI Attention LSTM  1.030958  1.409209  1.619687       0.902633
1  Zero Displacement Baseline  1.296014  1.582445  2.034650       1.582743
2  Constant Velocity Baseline  1.590166  2.184079  2.495407       1.266093
3        No-POI LSTM Baseline  1.184548  1.578481  1.862975       1.059850


## Round 2: Exclude Environment B

In [11]:
# Data Splitting for Round 2
test_env = 'B'

train_df_2 = df_final[df_final['env'] != test_env]
test_df_2  = df_final[df_final['env'] == test_env]

In [12]:
# Generate timestep sequences
X_train2, y_train2 = create_sequences_grouped(train_df_2, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
X_test2, y_test2 = create_sequences_grouped(test_df_2, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
print(f"X_train shape: {X_train2.shape}, y_train shape: {y_train2.shape}")
print(f"X_test shape: {X_test2.shape}, y_test shape: {y_test2.shape}")

X_train shape: (188749, 50, 28), y_train shape: (188749, 2)
X_test shape: (47497, 50, 28), y_test shape: (47497, 2)


### POI Model Construction

In [13]:
# Build an LSTM model that encodes each POI via a shared MLP and mean-pools POI context
inputs_2 = Input(shape=(HISTORY_LENGTH, 28), name='trajectory_input')

# Split user trajectory and POI features (skip userID at index 0)
user_features_2 = Lambda(lambda x: x[:, :, :3], name='user_features')(inputs_2)  # vx, vz, yaw

# Suppose you have 5 POIs, each with 1+ features (here, 9 features for all POIs)
# Reshape to (batch, HISTORY_LENGTH, 5, 1+poi_feat_dim) if needed
poi_features_2 = Lambda(lambda x: x[:, :, 3:], name='poi_raw')(inputs_2)
poi_reshaped_2 = Reshape((HISTORY_LENGTH, 5, 5), name='poi_reshape')(poi_features_2)  # Adjust as needed

# Shared MLP for each POI
poi_encoded_2 = TimeDistributed(TimeDistributed(Dense(32, activation='relu')), name='poi_mlp_1')(poi_reshaped_2)
poi_encoded_2 = TimeDistributed(TimeDistributed(Dense(16, activation='relu')), name='poi_mlp_2')(poi_encoded_2)

# Distance-biased attention pooling across the 5 POIs
poi_scores_2 = TimeDistributed(
    TimeDistributed(Dense(1)),
    name='learned_poi_scores'
)(poi_encoded_2)

poi_distances_2 = Lambda(
    lambda x: x[:, :, :, 0:1],
    name='poi_distances'
)(poi_reshaped_2)

distance_bias_2 = Lambda(
    lambda d: -d,
    name='distance_bias'
)(poi_distances_2)

combined_scores_2 = Lambda(
    lambda x: x[0] + x[1],
    name='combined_attention_scores'
)([poi_scores_2, distance_bias_2])

poi_weights_2 = Softmax(axis=2, name='poi_attention_weights')(combined_scores_2)

weighted_pois_2 = Multiply(name='weighted_pois')([poi_encoded_2, poi_weights_2])

poi_context_2 = Lambda(
    lambda x: tf.reduce_sum(x, axis=2),
    name='poi_attention_pool'
)(weighted_pois_2)

# Concatenate the POI context with user trajectory features
fused_2 = Concatenate(axis=-1, name='trajectory_poi_fusion')([user_features_2, poi_context_2])

# Process fused sequence through LSTM stack
x_2 = LSTM(64, return_sequences=True, name='lstm_1')(fused_2)
x_2 = LSTM(64, name='lstm_2')(x_2)
x_2 = Dropout(0.3, name='dropout')(x_2)
outputs_2 = Dense(2, name='position_output')(x_2)

# Build model
model_2= tf.keras.Model(inputs=inputs_2, outputs=outputs_2, name='poicontext_lstm')

model_2.summary()

Model: "poicontext_lstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ trajectory_input    │ (None, 50, 28)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ poi_raw (Lambda)    │ (None, 50, 25)    │          0 │ trajectory_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ poi_reshape         │ (None, 50, 5, 5)  │          0 │ poi_raw[0][0]     │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ poi_mlp_1           │ (None, 50, 5, 32) │        192 │ poi_reshape[0][0] │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ poi_mlp_2           │ (None, 50, 5, 16) │        528 │ poi_mlp_1[0][0]   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ poi_distances       │ (None, 50, 5, 1)  │          0 │ poi_reshape[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ learned_poi_scores  │ (None, 50, 5, 1)  │         17 │ poi_mlp_2[0][0]   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ distance_bias       │ (None, 50, 5, 1)  │          0 │ poi_distances[0]… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_attention… │ (None, 50, 5, 1)  │          0 │ learned_poi_scor… │
│ (Lambda)            │                   │            │ distance_bias[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ poi_attention_weig… │ (None, 50, 5, 1)  │          0 │ combined_attenti… │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weighted_pois       │ (None, 50, 5, 16) │          0 │ poi_mlp_2[0][0],  │
│ (Multiply)          │                   │            │ poi_attention_we… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_features       │ (None, 50, 3)     │          0 │ trajectory_input… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ poi_attention_pool  │ (None, 50, 16)    │          0 │ weighted_pois[0]… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ trajectory_poi_fus… │ (None, 50, 19)    │          0 │ user_features[0]… │
│ (Concatenate)       │                   │            │ poi_attention_po… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 50, 64)    │     21,504 │ trajectory_poi_f… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     33,024 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_output     │ (None, 2)         │        130 │ dropout[0][0]     │
│ (Dense)             │                   │            │                 

 Total params: 55,395 (216.39 KB)

 Trainable params: 55,395 (216.39 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### POI Model Training

In [ ]:
# Train the model
model_2.fit(
    X_train2, y_train2,
    epochs=100,
    batch_size=64,
    validation_data=(X_test2, y_test2)
)

In [ ]:
model_2.save("Exclude_Env_B_POI_Model.keras");

### POI Model Testing

In [15]:
# Make predictions for Round 2
preds_2 = model_2.predict(X_test2)

1485/1485 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step


In [16]:
# Baseline 1: zero displacement
preds_zero_2 = np.zeros_like(y_test2)

# Baseline 2: constant velocity
dt_2 = 0.05
horizon_seconds_2 = PREDICTION_HORIZON * dt_2

last_v_2 = X_test2[:, -1, :2]  # final vx, vz from input sequence
preds_cv_2 = last_v_2 * horizon_seconds_2

In [17]:
results_2 = []

results_2.append(evaluate_predictions("POI Attention LSTM", y_test2, preds_2))
results_2.append(evaluate_predictions("Zero Displacement Baseline", y_test2, preds_zero_2))
results_2.append(evaluate_predictions("Constant Velocity Baseline", y_test2, preds_cv_2))


POI Attention LSTM
MAE X: 1.2851, Z: 1.3165
RMSE X: 1.5606, Z: 1.5961
Overall MAE: 1.3008
Overall RMSE: 1.5783
MDE: 2.0472
Angular Error: 1.4012 radians

Zero Displacement Baseline
MAE X: 1.2964, Z: 1.3194
RMSE X: 1.5716, Z: 1.5982
Overall MAE: 1.3079
Overall RMSE: 1.5849
MDE: 2.0577
Angular Error: 1.5807 radians

Constant Velocity Baseline
MAE X: 1.5243, Z: 1.6501
RMSE X: 2.1120, Z: 2.3447
Overall MAE: 1.5872
Overall RMSE: 2.2284
MDE: 2.4865
Angular Error: 1.2630 radians


### Evaluate POI Model Performance

In [ ]:
# Analyze POI data validity
import matplotlib.pyplot as plt

poi_dist_cols_2 = [col for col in df_final.columns if 'poi' in col and 'dist' in col and 'weight' not in col]
poi_sin_cols_2 = [col for col in df_final.columns if 'poi' in col and 'sin_angle' in col]
poi_cos_cols_2 = [col for col in df_final.columns if 'poi' in col and 'cos_angle' in col]

print("POI Distance Statistics:")
print(df_final[poi_dist_cols_2].describe())

print("\nPOI Sin Angle Statistics:")
print(df_final[poi_sin_cols_2].describe())

print("\nPOI Cos Angle Statistics:")
print(df_final[poi_cos_cols_2].describe())

# Check correlation between yaw and POI features
yaw_col_2 = 'yaw'

corrs_yaw_sin_2 = df_final[[yaw_col_2] + poi_sin_cols_2].corr()
corrs_yaw_cos_2 = df_final[[yaw_col_2] + poi_cos_cols_2].corr()

print("\nCorrelation between yaw and POI sin(angle):")
print(corrs_yaw_sin_2[yaw_col_2])

print("\nCorrelation between yaw and POI cos(angle):")
print(corrs_yaw_cos_2[yaw_col_2])

# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, col in enumerate(poi_dist_cols_2[:3]):
    axes[0, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[0, i].set_title(f'{col} Distribution')
    axes[0, i].set_xlabel('Distance')
    axes[0, i].set_ylabel('Frequency')

for i, col in enumerate(poi_sin_cols_2[:3]):
    axes[1, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[1, i].set_title(f'{col} Distribution')
    axes[1, i].set_xlabel('sin(angle)')
    axes[1, i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Check ranges
print(f"\nSample sin(angle): {df_final[poi_sin_cols_2[0]].head()}")
print(f"sin(angle) range: {df_final[poi_sin_cols_2[0]].min():.3f} to {df_final[poi_sin_cols_2[0]].max():.3f}")

print(f"\nSample cos(angle): {df_final[poi_cos_cols_2[0]].head()}")
print(f"cos(angle) range: {df_final[poi_cos_cols_2[0]].min():.3f} to {df_final[poi_cos_cols_2[0]].max():.3f}")

print(f"\nYaw range: {df_final[yaw_col_2].min():.3f} to {df_final[yaw_col_2].max():.3f}")

In [18]:
# Check correlations between POI features and target positions
poi_cols2 = [col for col in df_final.columns if 'poi' in col]
target_cols2 = ['x', 'z']

corrs2 = df_final[poi_cols2 + target_cols2].corr()
print("Correlations between POI features and positions:")
print(corrs2.loc[poi_cols2, target_cols2])

# Also check with velocities and yaw
all_features2 = ['x', 'z', 'vx', 'vz', 'yaw'] + poi_cols2
corrs_full2 = df_final[all_features2].corr()
print("\nFull feature correlations with x and z:")
print(corrs_full2.loc[all_features2, ['x', 'z']])

# Check if POI features have predictive power
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Simple linear regression for each POI feature to x
for col in poi_cols2:
    X_feat_2 = df_final[[col]]
    y_x_2 = df_final['x']
    model_x_2 = LinearRegression().fit(X_feat_2, y_x_2)
    r2_x_2 = r2_score(y_x_2, model_x_2.predict(X_feat_2))
    
    y_z_2 = df_final['z']
    model_z_2 = LinearRegression().fit(X_feat_2, y_z_2)
    r2_z_2 = r2_score(y_z_2, model_z_2.predict(X_feat_2))
    
    print(f"{col}: R² with x = {r2_x_2:.4f}, with z = {r2_z_2:.4f}")

Correlations between POI features and positions:
                          x         z
poi_0_dist         0.004100 -0.001602
poi_0_sin_angle    0.001689 -0.055703
poi_0_cos_angle   -0.024572  0.071972
poi_0_is_in_front -0.021069  0.065554
poi_0_dist_weight  0.003595 -0.003493
poi_1_dist        -0.014937 -0.026769
poi_1_sin_angle    0.109229 -0.050508
poi_1_cos_angle   -0.029263  0.011836
poi_1_is_in_front -0.034826 -0.010136
poi_1_dist_weight -0.016935 -0.035752
poi_2_dist        -0.013574 -0.043339
poi_2_sin_angle    0.100123 -0.055151
poi_2_cos_angle   -0.046227  0.037010
poi_2_is_in_front -0.041342  0.001516
poi_2_dist_weight -0.012337 -0.041651
poi_3_dist        -0.013234 -0.030513
poi_3_sin_angle    0.066311 -0.050219
poi_3_cos_angle   -0.053650  0.026235
poi_3_is_in_front -0.045093  0.003115
poi_3_dist_weight -0.011307 -0.023909
poi_4_dist         0.002375  0.002682
poi_4_sin_angle    0.055533 -0.047950
poi_4_cos_angle   -0.049235  0.020808
poi_4_is_in_front -0.031860  0.018500
p

### Baseline Model Construction

In [19]:
# Features without POIs: vx, vz, yaw (3 features)
X_train_no_poi_2 = X_train2[:, :, :3]
X_test_no_poi_2 = X_test2[:, :, :3]

model_no_poi_2 = Sequential([
    LSTM(64, input_shape=(HISTORY_LENGTH, 3), return_sequences=True),
    LSTM(64),
    Dropout(0.3),
    Dense(2)
])

model_no_poi_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

c:\Users\JetPa\anaconda3\envs\mnist_tf2\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Baseline Model Training

In [ ]:
# Training baseline model for Round 2
history_no_poi_2 = model_no_poi_2.fit(
    X_train_no_poi_2, y_train2,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_no_poi_2, y_test2),
    verbose=0
)

In [ ]:
model_no_poi_2.save("Exclude_Env_B_Baseline_Model.keras");

### Baseline Model Testing

In [21]:
preds_no_poi2 = model_no_poi_2.predict(X_test_no_poi_2)

results_2.append(evaluate_predictions("No-POI LSTM Baseline", y_test2, preds_no_poi2))

results_df2 = pd.DataFrame(results_2)
print(results_df2)

1485/1485 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step

No-POI LSTM Baseline
MAE X: 1.2950, Z: 1.3128
RMSE X: 1.5715, Z: 1.5929
Overall MAE: 1.3039
Overall RMSE: 1.5822
MDE: 2.0512
Angular Error: 1.4202 radians
                        Model       MAE      RMSE       MDE  Angular_Error
0          POI Attention LSTM  1.300816  1.578346  2.047177       1.401195
1  Zero Displacement Baseline  1.307910  1.584878  2.057680       1.580653
2  Constant Velocity Baseline  1.587170  2.228350  2.486478       1.262987
3        No-POI LSTM Baseline  1.303890  1.582209  2.051182       1.420232


## Round 3: Exclude Environment C

In [ ]:
# Data Splitting for Round 3
test_env = 'C'

train_df_3 = df_final[df_final['env'] != test_env]
test_df_3  = df_final[df_final['env'] == test_env]

In [ ]:
# Generate timestep sequences for Round 3
X_train_3, y_train_3 = create_sequences_grouped(train_df_3, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
X_test_3, y_test_3 = create_sequences_grouped(test_df_3, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
print(f"X_train_3 shape: {X_train_3.shape}, y_train_3 shape: {y_train_3.shape}")
print(f"X_test_3 shape: {X_test_3.shape}, y_test_3 shape: {y_test_3.shape}")

### POI Model Construction

In [ ]:
# Build an LSTM model that encodes each POI via a shared MLP and mean-pools POI context
inputs_3 = Input(shape=(HISTORY_LENGTH, 28), name='trajectory_input')

# Split user trajectory and POI features (skip userID at index 0)
user_features_3 = Lambda(lambda x: x[:, :, :3], name='user_features')(inputs_3)  # vx, vz, yaw

# Suppose you have 5 POIs, each with 1+ features (here, 9 features for all POIs)
# Reshape to (batch, HISTORY_LENGTH, 5, 1+poi_feat_dim) if needed
poi_features_3 = Lambda(lambda x: x[:, :, 3:], name='poi_raw')(inputs_3)
poi_reshaped_3 = Reshape((HISTORY_LENGTH, 5, 5), name='poi_reshape')(poi_features_3)  # Adjust as needed

# Shared MLP for each POI
poi_encoded_3 = TimeDistributed(TimeDistributed(Dense(32, activation='relu')), name='poi_mlp_1')(poi_reshaped_3)
poi_encoded_3 = TimeDistributed(TimeDistributed(Dense(16, activation='relu')), name='poi_mlp_2')(poi_encoded_3)

# Distance-biased attention pooling across the 5 POIs
poi_scores_3 = TimeDistributed(
    TimeDistributed(Dense(1)),
    name='learned_poi_scores'
)(poi_encoded_3)

poi_distances_3 = Lambda(
    lambda x: x[:, :, :, 0:1],
    name='poi_distances'
)(poi_reshaped_3)

distance_bias_3 = Lambda(
    lambda d: -d,
    name='distance_bias'
)(poi_distances_3)

combined_scores_3 = Lambda(
    lambda x: x[0] + x[1],
    name='combined_attention_scores'
)([poi_scores_3, distance_bias_3])

poi_weights_3 = Softmax(axis=2, name='poi_attention_weights')(combined_scores_3)

weighted_pois_3 = Multiply(name='weighted_pois')([poi_encoded_3, poi_weights_3])

poi_context_3 = Lambda(
    lambda x: tf.reduce_sum(x, axis=2),
    name='poi_attention_pool'
)(weighted_pois_3)

# Concatenate the POI context with user trajectory features
fused_3 = Concatenate(axis=-1, name='trajectory_poi_fusion')([user_features_3, poi_context_3])

# Process fused sequence through LSTM stack
x_3 = LSTM(64, return_sequences=True, name='lstm_1')(fused_3)
x_3 = LSTM(64, name='lstm_2')(x_3)
x_3 = Dropout(0.3, name='dropout')(x_3)
outputs_3 = Dense(2, name='position_output')(x_3)

# Build model
model_3 = tf.keras.Model(inputs=inputs_3, outputs=outputs_3, name='poicontext_lstm')

model_3.summary()

In [ ]:
model_3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### POI Model Training

In [ ]:
# Train the model for Round 3
model_3.fit(
    X_train_3, y_train_3,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_3, y_test_3)
)

In [ ]:
model_3.save("Exclude_Env_C_POI_Model.keras");

### POI Model Testing

In [ ]:
# Make predictions for Round 3
preds_3 = model_3.predict(X_test_3)

In [ ]:
# Baseline 1: zero displacement
preds_zero_3 = np.zeros_like(y_test_3)

# Baseline 2: constant velocity
dt_3 = 0.05
horizon_seconds_3 = PREDICTION_HORIZON * dt_3

last_v_3 = X_test_3[:, -1, :2]  # final vx, vz from input sequence
preds_cv_3 = last_v_3 * horizon_seconds_3

In [ ]:
results_3 = []

results_3.append(evaluate_predictions("POI Attention LSTM", y_test_3, preds_3))
results_3.append(evaluate_predictions("Zero Displacement Baseline", y_test_3, preds_zero_3))
results_3.append(evaluate_predictions("Constant Velocity Baseline", y_test_3, preds_cv_3))

### Evaluate POI Model Performance

In [ ]:
# Analyze POI data validity
import matplotlib.pyplot as plt

poi_dist_cols_3 = [col for col in df_final.columns if 'poi' in col and 'dist' in col and 'weight' not in col]
poi_sin_cols_3 = [col for col in df_final.columns if 'poi' in col and 'sin_angle' in col]
poi_cos_cols_3 = [col for col in df_final.columns if 'poi' in col and 'cos_angle' in col]

print("POI Distance Statistics:")
print(df_final[poi_dist_cols_3].describe())

print("\nPOI Sin Angle Statistics:")
print(df_final[poi_sin_cols_3].describe())

print("\nPOI Cos Angle Statistics:")
print(df_final[poi_cos_cols_3].describe())

# Check correlation between yaw and POI features
yaw_col_3 = 'yaw'

corrs_yaw_sin_3 = df_final[[yaw_col_3] + poi_sin_cols_3].corr()
corrs_yaw_cos_3 = df_final[[yaw_col_3] + poi_cos_cols_3].corr()

print("\nCorrelation between yaw and POI sin(angle):")
print(corrs_yaw_sin_3[yaw_col_3])

print("\nCorrelation between yaw and POI cos(angle):")
print(corrs_yaw_cos_3[yaw_col_3])

# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, col in enumerate(poi_dist_cols_3[:3]):
    axes[0, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[0, i].set_title(f'{col} Distribution')
    axes[0, i].set_xlabel('Distance')
    axes[0, i].set_ylabel('Frequency')

for i, col in enumerate(poi_sin_cols_3[:3]):
    axes[1, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[1, i].set_title(f'{col} Distribution')
    axes[1, i].set_xlabel('sin(angle)')
    axes[1, i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Check ranges
print(f"\nSample sin(angle): {df_final[poi_sin_cols_3[0]].head()}")
print(f"sin(angle) range: {df_final[poi_sin_cols_3[0]].min():.3f} to {df_final[poi_sin_cols_3[0]].max():.3f}")

print(f"\nSample cos(angle): {df_final[poi_cos_cols_3[0]].head()}")
print(f"cos(angle) range: {df_final[poi_cos_cols_3[0]].min():.3f} to {df_final[poi_cos_cols_3[0]].max():.3f}")

print(f"\nYaw range: {df_final[yaw_col_3].min():.3f} to {df_final[yaw_col_3].max():.3f}")

In [ ]:
# Check correlations between POI features and target positions
poi_cols_3 = [col for col in df_final.columns if 'poi' in col]
target_cols_3 = ['x', 'z']

corrs_3 = df_final[poi_cols_3 + target_cols_3].corr()
print("Correlations between POI features and positions:")
print(corrs_3.loc[poi_cols_3, target_cols_3])

# Also check with velocities and yaw
all_features_3 = ['x', 'z', 'vx', 'vz', 'yaw'] + poi_cols_3
corrs_full_3 = df_final[all_features_3].corr()
print("\nFull feature correlations with x and z:")
print(corrs_full_3.loc[all_features_3, ['x', 'z']])

# Check if POI features have predictive power
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Simple linear regression for each POI feature to x
for col in poi_cols_3:
    X_feat_3 = df_final[[col]]
    y_x_3 = df_final['x']
    model_x_3 = LinearRegression().fit(X_feat_3, y_x_3)
    r2_x_3 = r2_score(y_x_3, model_x_3.predict(X_feat_3))
    
    y_z_3 = df_final['z']
    model_z_3 = LinearRegression().fit(X_feat_3, y_z_3)
    r2_z_3 = r2_score(y_z_3, model_z_3.predict(X_feat_3))
    
    print(f"{col}: R² with x = {r2_x_3:.4f}, with z = {r2_z_3:.4f}")

### Baseline Model Construction

In [ ]:
# Features without POIs: vx, vz, yaw (3 features)
X_train_no_poi_3 = X_train_3[:, :, :3]
X_test_no_poi_3 = X_test_3[:, :, :3]

model_no_poi_3 = Sequential([
    LSTM(64, input_shape=(HISTORY_LENGTH, 3), return_sequences=True),
    LSTM(64),
    Dropout(0.3),
    Dense(2)
])

model_no_poi_3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### Baseline Model Training

In [ ]:
# Training baseline model for Round 3
history_no_poi_3 = model_no_poi_3.fit(
    X_train_no_poi_3, y_train_3,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_no_poi_3, y_test_3),
    verbose=0
)

In [ ]:
model_no_poi_3.save("Exclude_Env_C_Baseline_Model.keras");

### POI Model Testing

In [ ]:
preds_no_poi_3 = model_no_poi_3.predict(X_test_no_poi_3)

results_3.append(evaluate_predictions("No-POI LSTM Baseline", y_test, preds_no_poi_3))

results_df_3 = pd.DataFrame(results_3)
print(results_df_3)

## Round 4: Exclude Environment D

In [ ]:
# Data Splitting for Round 4
test_env = 'D'

train_df_4 = df_final[df_final['env'] != test_env]
test_df_4  = df_final[df_final['env'] == test_env]

In [ ]:
# Generate timestep sequences for Round 4
X_train_4, y_train_4 = create_sequences_grouped(train_df_4, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
X_test_4, y_test_4 = create_sequences_grouped(test_df_4, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
print(f"X_train_4 shape: {X_train_4.shape}, y_train_4 shape: {y_train_4.shape}")
print(f"X_test_4 shape: {X_test_4.shape}, y_test_4 shape: {y_test_4.shape}")

### POI Model Construction

In [ ]:
# Build an LSTM model that encodes each POI via a shared MLP and mean-pools POI context
inputs_4 = Input(shape=(HISTORY_LENGTH, 28), name='trajectory_input')

# Split user trajectory and POI features (skip userID at index 0)
user_features_4 = Lambda(lambda x: x[:, :, :3], name='user_features')(inputs_4)  # vx, vz, yaw

# Suppose you have 5 POIs, each with 1+ features (here, 9 features for all POIs)
# Reshape to (batch, HISTORY_LENGTH, 5, 1+poi_feat_dim) if needed
poi_features_4 = Lambda(lambda x: x[:, :, 3:], name='poi_raw')(inputs_4)
poi_reshaped_4 = Reshape((HISTORY_LENGTH, 5, 5), name='poi_reshape')(poi_features_4)  # Adjust as needed

# Shared MLP for each POI
poi_encoded_4 = TimeDistributed(TimeDistributed(Dense(32, activation='relu')), name='poi_mlp_1')(poi_reshaped_4)
poi_encoded_4 = TimeDistributed(TimeDistributed(Dense(16, activation='relu')), name='poi_mlp_2')(poi_encoded_4)

# Distance-biased attention pooling across the 5 POIs
poi_scores_4 = TimeDistributed(
    TimeDistributed(Dense(1)),
    name='learned_poi_scores'
)(poi_encoded_4)

poi_distances_4 = Lambda(
    lambda x: x[:, :, :, 0:1],
    name='poi_distances'
)(poi_reshaped_4)

distance_bias_4 = Lambda(
    lambda d: -d,
    name='distance_bias'
)(poi_distances_4)

combined_scores_4 = Lambda(
    lambda x: x[0] + x[1],
    name='combined_attention_scores'
)([poi_scores_4, distance_bias_4])

poi_weights_4 = Softmax(axis=2, name='poi_attention_weights')(combined_scores_4)

weighted_pois_4 = Multiply(name='weighted_pois')([poi_encoded_4, poi_weights_4])

poi_context_4 = Lambda(
    lambda x: tf.reduce_sum(x, axis=2),
    name='poi_attention_pool'
)(weighted_pois_4)

# Concatenate the POI context with user trajectory features
fused_4 = Concatenate(axis=-1, name='trajectory_poi_fusion')([user_features_4, poi_context_4])

# Process fused sequence through LSTM stack
x_4 = LSTM(64, return_sequences=True, name='lstm_1')(fused_4)
x_4 = LSTM(64, name='lstm_2')(x_4)
x_4 = Dropout(0.3, name='dropout')(x_4)
outputs_4 = Dense(2, name='position_output')(x_4)

# Build model
model_4 = tf.keras.Model(inputs=inputs_4, outputs=outputs_4, name='poicontext_lstm')

model_4.summary()

In [ ]:
model_4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### POI Model Training

In [ ]:
# Train the model for Round 4
model_4.fit(
    X_train_4, y_train_4,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_4, y_test_4)
)

In [ ]:
model_4.save("Exclude_Env_D_POI_Model.keras");

### POI Model Testing

In [ ]:
# Make predictions for Round 4
preds_4 = model_4.predict(X_test_4)

In [ ]:
# Baseline 1: zero displacement
preds_zero_4 = np.zeros_like(y_test_4)

# Baseline 2: constant velocity
dt_4 = 0.05
horizon_seconds_4 = PREDICTION_HORIZON * dt_4

last_v_4 = X_test_4[:, -1, :2]  # final vx, vz from input sequence
preds_cv_4 = last_v_4 * horizon_seconds_4

In [ ]:
results_4 = []

results_4.append(evaluate_predictions("POI Attention LSTM", y_test_4, preds_4))
results_4.append(evaluate_predictions("Zero Displacement Baseline", y_test_4, preds_zero_4))
results_4.append(evaluate_predictions("Constant Velocity Baseline", y_test_4, preds_cv_4))

### Evaluate POI Model Performance

In [ ]:
# Analyze POI data validity
import matplotlib.pyplot as plt

poi_dist_cols_4 = [col for col in df_final.columns if 'poi' in col and 'dist' in col and 'weight' not in col]
poi_sin_cols_4 = [col for col in df_final.columns if 'poi' in col and 'sin_angle' in col]
poi_cos_cols_4 = [col for col in df_final.columns if 'poi' in col and 'cos_angle' in col]

print("POI Distance Statistics:")
print(df_final[poi_dist_cols_4].describe())

print("\nPOI Sin Angle Statistics:")
print(df_final[poi_sin_cols_4].describe())

print("\nPOI Cos Angle Statistics:")
print(df_final[poi_cos_cols_4].describe())

# Check correlation between yaw and POI features
yaw_col_4 = 'yaw'

corrs_yaw_sin_4 = df_final[[yaw_col_4] + poi_sin_cols_4].corr()
corrs_yaw_cos_4 = df_final[[yaw_col_4] + poi_cos_cols_4].corr()

print("\nCorrelation between yaw and POI sin(angle):")
print(corrs_yaw_sin_4[yaw_col_4])

print("\nCorrelation between yaw and POI cos(angle):")
print(corrs_yaw_cos_4[yaw_col_4])

# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, col in enumerate(poi_dist_cols_4[:3]):
    axes[0, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[0, i].set_title(f'{col} Distribution')
    axes[0, i].set_xlabel('Distance')
    axes[0, i].set_ylabel('Frequency')

for i, col in enumerate(poi_sin_cols_4[:3]):
    axes[1, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[1, i].set_title(f'{col} Distribution')
    axes[1, i].set_xlabel('sin(angle)')
    axes[1, i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Check ranges
print(f"\nSample sin(angle): {df_final[poi_sin_cols_4[0]].head()}")
print(f"sin(angle) range: {df_final[poi_sin_cols_4[0]].min():.3f} to {df_final[poi_sin_cols_4[0]].max():.3f}")

print(f"\nSample cos(angle): {df_final[poi_cos_cols_4[0]].head()}")
print(f"cos(angle) range: {df_final[poi_cos_cols_4[0]].min():.3f} to {df_final[poi_cos_cols_4[0]].max():.3f}")

print(f"\nYaw range: {df_final[yaw_col_4].min():.3f} to {df_final[yaw_col_4].max():.3f}")

In [ ]:
# Check correlations between POI features and target positions
poi_cols_4 = [col for col in df_final.columns if 'poi' in col]
target_cols_4 = ['x', 'z']

corrs_4 = df_final[poi_cols_4 + target_cols_4].corr()
print("Correlations between POI features and positions:")
print(corrs_4.loc[poi_cols_4, target_cols_4])

# Also check with velocities and yaw
all_features_4 = ['x', 'z', 'vx', 'vz', 'yaw'] + poi_cols_4
corrs_full_4 = df_final[all_features_4].corr()
print("\nFull feature correlations with x and z:")
print(corrs_full_4.loc[all_features_4, ['x', 'z']])

# Check if POI features have predictive power
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Simple linear regression for each POI feature to x
for col in poi_cols_4:
    X_feat_4 = df_final[[col]]
    y_x_4 = df_final['x']
    model_x_4 = LinearRegression().fit(X_feat_4, y_x_4)
    r2_x_4 = r2_score(y_x_4, model_x_4.predict(X_feat_4))
    
    y_z_4 = df_final['z']
    model_z_4 = LinearRegression().fit(X_feat_4, y_z_4)
    r2_z_4 = r2_score(y_z_4, model_z_4.predict(X_feat_4))
    
    print(f"{col}: R² with x = {r2_x_4:.4f}, with z = {r2_z_4:.4f}")

### Baseline Model Construction

In [ ]:
# Features without POIs: vx, vz, yaw (3 features)
X_train_no_poi_4 = X_train_4[:, :, :3]
X_test_no_poi_4 = X_test_4[:, :, :3]

model_no_poi_4 = Sequential([
    LSTM(64, input_shape=(HISTORY_LENGTH, 3), return_sequences=True),
    LSTM(64),
    Dropout(0.3),
    Dense(2)
])

model_no_poi_4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### Baseline Model Training

In [ ]:
# Training baseline model for Round 4
history_no_poi_4 = model_no_poi_4.fit(
    X_train_no_poi_4, y_train_4,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_no_poi_4, y_test_4),
    verbose=0
)

In [ ]:
model_no_poi_4.save("Exclude_Env_D_Baseline_Model.keras");

### Baseline Model Testing

In [ ]:
preds_no_poi_4 = model_no_poi_4.predict(X_test_no_poi_4)

results.append(evaluate_predictions("No-POI LSTM Baseline", y_test_4, preds_no_poi_4))

results_df_4 = pd.DataFrame(results)
print(results_df_4)

## Round 5: Exclude Two Users (Inter-User Performance)

In [ ]:
# Data Splitting for Round 5
test_users = [2, 15]  # Exclude users 2 and 15 for testing

train_df_5 = df_final[~df_final['userID'].isin(test_users)]
test_df_5 = df_final[df_final['userID'].isin(test_users)]

In [ ]:
# Generate timestep sequences for Round 5
X_train_5, y_train_5 = create_sequences_grouped(train_df_5, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
X_test_5, y_test_5 = create_sequences_grouped(test_df_5, seq_length=HISTORY_LENGTH, prediction_horizon=PREDICTION_HORIZON)
print(f"X_train_5 shape: {X_train_5.shape}, y_train_5 shape: {y_train_5.shape}")
print(f"X_test_5 shape: {X_test_5.shape}, y_test_5 shape: {y_test_5.shape}")

### POI Model Construction

In [ ]:
# Build an LSTM model that encodes each POI via a shared MLP and mean-pools POI context
inputs_5 = Input(shape=(HISTORY_LENGTH, 28), name='trajectory_input')

# Split user trajectory and POI features (skip userID at index 0)
user_features_5 = Lambda(lambda x: x[:, :, :3], name='user_features')(inputs_5)  # vx, vz, yaw

# Suppose you have 5 POIs, each with 1+ features (here, 9 features for all POIs)
# Reshape to (batch, HISTORY_LENGTH, 5, 1+poi_feat_dim) if needed
poi_features_5 = Lambda(lambda x: x[:, :, 3:], name='poi_raw')(inputs_5)
poi_reshaped_5 = Reshape((HISTORY_LENGTH, 5, 5), name='poi_reshape')(poi_features_5)  # Adjust as needed

# Shared MLP for each POI
poi_encoded_5 = TimeDistributed(TimeDistributed(Dense(32, activation='relu')), name='poi_mlp_1')(poi_reshaped_5)
poi_encoded_5 = TimeDistributed(TimeDistributed(Dense(16, activation='relu')), name='poi_mlp_2')(poi_encoded_5)

# Distance-biased attention pooling across the 5 POIs
poi_scores_5 = TimeDistributed(
    TimeDistributed(Dense(1)),
    name='learned_poi_scores'
)(poi_encoded_5)

poi_distances_5 = Lambda(
    lambda x: x[:, :, :, 0:1],
    name='poi_distances'
)(poi_reshaped_5)

distance_bias_5 = Lambda(
    lambda d: -d,
    name='distance_bias'
)(poi_distances_5)

combined_scores_5 = Lambda(
    lambda x: x[0] + x[1],
    name='combined_attention_scores'
)([poi_scores_5, distance_bias_5])

poi_weights_5 = Softmax(axis=2, name='poi_attention_weights')(combined_scores_5)

weighted_pois_5 = Multiply(name='weighted_pois')([poi_encoded_5, poi_weights_5])

poi_context_5 = Lambda(
    lambda x: tf.reduce_sum(x, axis=2),
    name='poi_attention_pool'
)(weighted_pois_5)

# Concatenate the POI context with user trajectory features
fused = Concatenate(axis=-1, name='trajectory_poi_fusion')([user_features_5, poi_context_5])

# Process fused sequence through LSTM stack
x_5 = LSTM(64, return_sequences=True, name='lstm_1')(fused)
x_5 = LSTM(64, name='lstm_2')(x_5)
x_5 = Dropout(0.3, name='dropout')(x_5)
outputs_5 = Dense(2, name='position_output')(x_5)

# Build model
model_5 = tf.keras.Model(inputs=inputs_5, outputs=outputs_5, name='poicontext_lstm')

model_5.summary()

In [ ]:
model_5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### POI Model Training

In [ ]:
# Train the model for Round 5
model_5.fit(
    X_train_5, y_train_5,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_5, y_test_5)
)

In [ ]:
model_5.save("Exclude_Users_POI_Model.keras");

### POI Model Testing

In [ ]:
# Make predictions for Round 5
preds_5 = model_5.predict(X_test_5)

In [ ]:
# Baseline 1: zero displacement
preds_zero_5 = np.zeros_like(y_test_5)

# Baseline 2: constant velocity
dt_5 = 0.05
horizon_seconds_5 = PREDICTION_HORIZON * dt_5

last_v_5 = X_test_5[:, -1, :2]  # final vx, vz from input sequence
preds_cv_5 = last_v_5 * horizon_seconds_5

In [ ]:
results_5 = []

results_5.append(evaluate_predictions("POI Attention LSTM", y_test_5, preds_5))
results_5.append(evaluate_predictions("Zero Displacement Baseline", y_test_5, preds_zero_5))
results_5.append(evaluate_predictions("Constant Velocity Baseline", y_test_5, preds_cv_5))

### Evaluate POI Model Performance

In [ ]:
# Analyze POI data validity
import matplotlib.pyplot as plt

poi_dist_cols_5 = [col for col in df_final.columns if 'poi' in col and 'dist' in col and 'weight' not in col]
poi_sin_cols_5 = [col for col in df_final.columns if 'poi' in col and 'sin_angle' in col]
poi_cos_cols_5 = [col for col in df_final.columns if 'poi' in col and 'cos_angle' in col]

print("POI Distance Statistics:")
print(df_final[poi_dist_cols_5].describe())

print("\nPOI Sin Angle Statistics:")
print(df_final[poi_sin_cols_5].describe())

print("\nPOI Cos Angle Statistics:")
print(df_final[poi_cos_cols_5].describe())

# Check correlation between yaw and POI features
yaw_col_5 = 'yaw'

corrs_yaw_sin_5 = df_final[[yaw_col_5] + poi_sin_cols_5].corr()
corrs_yaw_cos_5 = df_final[[yaw_col_5] + poi_cos_cols_5].corr()

print("\nCorrelation between yaw and POI sin(angle):")
print(corrs_yaw_sin_5[yaw_col_5])

print("\nCorrelation between yaw and POI cos(angle):")
print(corrs_yaw_cos_5[yaw_col_5])

# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, col in enumerate(poi_dist_cols_5[:3]):
    axes[0, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[0, i].set_title(f'{col} Distribution')
    axes[0, i].set_xlabel('Distance')
    axes[0, i].set_ylabel('Frequency')

for i, col in enumerate(poi_sin_cols_5[:3]):
    axes[1, i].hist(df_final[col], bins=50, alpha=0.7)
    axes[1, i].set_title(f'{col} Distribution')
    axes[1, i].set_xlabel('sin(angle)')
    axes[1, i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Check ranges
print(f"\nSample sin(angle): {df_final[poi_sin_cols_5[0]].head()}")
print(f"sin(angle) range: {df_final[poi_sin_cols_5[0]].min():.3f} to {df_final[poi_sin_cols_5[0]].max():.3f}")

print(f"\nSample cos(angle): {df_final[poi_cos_cols_5[0]].head()}")
print(f"cos(angle) range: {df_final[poi_cos_cols_5[0]].min():.3f} to {df_final[poi_cos_cols_5[0]].max():.3f}")

print(f"\nYaw range: {df_final[yaw_col_5].min():.3f} to {df_final[yaw_col_5].max():.3f}")

In [ ]:
# Check correlations between POI features and target positions
poi_cols_5 = [col for col in df_final.columns if 'poi' in col]
target_cols_5 = ['x', 'z']

corrs_5 = df_final[poi_cols_5 + target_cols_5].corr()
print("Correlations between POI features and positions:")
print(corrs_5.loc[poi_cols_5, target_cols_5])

# Also check with velocities and yaw
all_features_5 = ['x', 'z', 'vx', 'vz', 'yaw'] + poi_cols_5
corrs_full_5 = df_final[all_features_5].corr()
print("\nFull feature correlations with x and z:")
print(corrs_full_5.loc[all_features_5, ['x', 'z']])

# Check if POI features have predictive power
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Simple linear regression for each POI feature to x
for col in poi_cols_5:
    X_feat_5 = df_final[[col]]
    y_x_5 = df_final['x']
    model_x_5 = LinearRegression().fit(X_feat_5, y_x_5)
    r2_x_5 = r2_score(y_x_5, model_x_5.predict(X_feat_5))
    
    y_z_5 = df_final['z']
    model_z_5 = LinearRegression().fit(X_feat_5, y_z_5)
    r2_z_5 = r2_score(y_z_5, model_z_5.predict(X_feat_5))
    
    print(f"{col}: R² with x = {r2_x_5:.4f}, with z = {r2_z_5:.4f}")

### Baseline Model Construction

In [ ]:
# Features without POIs: vx, vz, yaw (3 features)
X_train_no_poi_5 = X_train_5[:, :, :3]
X_test_no_poi_5 = X_test_5[:, :, :3]

model_no_poi_5 = Sequential([
    LSTM(64, input_shape=(HISTORY_LENGTH, 3), return_sequences=True),
    LSTM(64),
    Dropout(0.3),
    Dense(2)
])

model_no_poi_5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='mse',
    metrics=['mae']
)

### Baseline Model Training

In [ ]:
# Training baseline model for Round 5 
history_no_poi_5 = model_no_poi_5.fit(
    X_train_no_poi_5, y_train_5,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_no_poi_5, y_test_5),
    verbose=0
)

In [ ]:
model_no_poi_5.save("Exclude_Users_Baseline_Model.keras");

### Baseline Model Testing

In [ ]:
preds_no_poi_5 = model_no_poi_5.predict(X_test_no_poi_5)

results_5.append(evaluate_predictions("No-POI LSTM Baseline", y_test_5, preds_no_poi_5))

results_df_5 = pd.DataFrame(results_5)
print(results_df_5)